# S2 results: α curves across datasets

Collects every finished S2 run under `OUT_DIR` (`config.json` + `metrics.json`; unfinished or unreadable
runs are skipped) into one table per dataset × arch × α with mean ± std over seeds and n, and plots the
test metric against α with the `mlp` and `gcn` baselines as horizontal lines. CSV and PNG files go to
`DRIVE/results/s2/`. **A CPU runtime is fine.** Safe to run while sweeps are still going.

## 1. Settings (edit)

`BRANCH` defaults to `main`; set it to another branch to run that code instead.

In [ ]:
# EDIT: settings for this notebook
BRANCH = 'social-pipeline'          # git branch to run; change it to test a feature branch
USE_TOKEN = False        # True if the repo is private (needs the GITHUB_TOKEN Colab secret, see step 3)
DRIVE = '/content/drive/MyDrive/Thesis/gnn_mech'
DATA_ROOT = f'{DRIVE}/data'    # downloads and processed caches (kept on Drive, reused by every session)
OUT_DIR = f'{DRIVE}/runs'      # run folders (kept on Drive so runs resume after a disconnect)
LOCAL_DATA = '/content/data'   # fast local copy of the dataset used during training

## 2. Runtime check (a CPU runtime is fine here)

In [ ]:
!nvidia-smi || echo 'no GPU: fine for this notebook'

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Get the code

Clones the repo at `BRANCH`, or pulls if it is already there. If the repo is private, create a GitHub
personal access token (read access to the repo is enough), store it as a Colab secret named
`GITHUB_TOKEN` (key icon in the left sidebar) and set `USE_TOKEN = True` in step 1. Never paste the
token into the notebook itself.

In [ ]:
import os, subprocess

REPO_DIR = '/content/message-passing-in-gnns'
url = 'https://github.com/djdhawal/message-passing-in-gnns.git'
if USE_TOKEN:
    from google.colab import userdata
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/djdhawal/message-passing-in-gnns.git"

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, url, REPO_DIR], check=True)
%cd {REPO_DIR}
!git log -1 --oneline

## 5. Install

In [ ]:
!pip install -q torch_geometric
!pip install -q -e .

## 6. Folders on Drive

In [ ]:
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)
print('data on Drive:', DATA_ROOT)
print('runs on Drive:', OUT_DIR)

## 7. Which experiments count as S2

In [ ]:
import glob

# exp_names come from the sweep files; if they cannot be read, every social run except the quickstart counts.
S2_EXPS = set()
for path in sorted(glob.glob('configs/sweeps/social_s2_*.yaml')):
    try:
        from gnn_mech.sweep import load_sweep
        S2_EXPS |= {r.cfg.exp_name for r in load_sweep(path, [f'out_dir={OUT_DIR}'])[0]}
    except Exception as e:
        print(f'[warn] {path}: {e!r}')
print('S2 experiments:', sorted(S2_EXPS) or '(none found: falling back to data.name filter)')

## 8. Collect runs

In [ ]:
import json, os
import numpy as np
import pandas as pd

PRIMARY = ('auroc', 'acc', 'r2', 'ap', 'mae', 'rmse')   # first one present is the headline metric
BASELINES = ('mlp', 'gcn')


def read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def primary_metric(split):
    keys = [k for k, v in (split or {}).items() if k != 'loss' and isinstance(v, (int, float))]
    for k in PRIMARY:
        if k in keys:
            return k
    return keys[0] if keys else None

def collect(out_dir, exps=None):
    # One row per finished run (config.json + metrics.json); unreadable or unfinished runs are skipped.
    rows = []
    root = Path(out_dir)
    if not root.is_dir():
        return pd.DataFrame()
    for cfg_path in root.glob('*/*/seed*/config.json'):
        run = cfg_path.parent
        exp = run.parent.parent.name
        if exps is not None and exp not in exps:
            continue
        cfg, m = read_json(cfg_path), read_json(run / 'metrics.json')
        if not cfg or not m or not isinstance(m.get('test'), dict):
            continue
        model, data = cfg.get('model', {}), cfg.get('data', {})
        metric = primary_metric(m['test'])
        row = {'exp': exp, 'hash': run.parent.name, 'dataset': data.get('name'), 'target': data.get('target', ''),
               'arch': model.get('arch'), 'attn': model.get('attn', 'softmax'), 'alpha': model.get('alpha'),
               'seed': cfg.get('seed'), 'metric': metric,
               'test': m['test'].get(metric), 'val': (m.get('val') or {}).get(metric),
               'best_epoch': m.get('best_epoch'), 'epochs_run': m.get('epochs_run'),
               'hours': (m.get('seconds') or 0) / 3600, 'n_params': m.get('n_params'), 'path': str(run)}
        row.update({f'test_{k}': v for k, v in m['test'].items() if isinstance(v, (int, float))})
        rows.append(row)
    return pd.DataFrame(rows)


def summarize(df):
    # dataset x arch x alpha: mean +- std over seeds and n. Baselines ignore alpha.
    if df.empty:
        return df
    df = df.copy()
    df['alpha'] = np.where(df['arch'].isin(BASELINES), np.nan, df['alpha'])
    metric_cols = [c for c in df.columns if c.startswith('test_')]
    g = df.groupby(['dataset', 'metric', 'arch', 'attn', 'alpha'], dropna=False)
    out = g.agg(n=('seed', 'nunique'), mean=('test', 'mean'), std=('test', 'std'),
                val_mean=('val', 'mean'), hours=('hours', 'sum'),
                **{f'{c}_mean': (c, 'mean') for c in metric_cols}).reset_index()
    out['std'] = out['std'].fillna(0.0)
    out['test (mean ± std)'] = [f'{a:.4f} ± {b:.4f}' for a, b in zip(out['mean'], out['std'])]
    return out.sort_values(['dataset', 'arch', 'alpha']).reset_index(drop=True)

from pathlib import Path

SOCIAL = {'twitch-gamers', 'reddit-body', 'higgs-follow', 'higgs-retweet', 'higgs-reply', 'higgs-mention',
          *(f'twitch-{l}' for l in ('de', 'en', 'es', 'fr', 'pt', 'ru'))}
runs_df = collect(OUT_DIR, sorted(S2_EXPS) if S2_EXPS else None)
if not S2_EXPS and not runs_df.empty:
    runs_df = runs_df[runs_df['dataset'].isin(SOCIAL) & ~runs_df['exp'].str.contains('quickstart')]
print(len(runs_df), 'finished runs')
if not runs_df.empty:
    display(runs_df.groupby(['dataset', 'exp']).size().rename('runs').reset_index())

## 9. Table: dataset × arch × α (test metric, mean ± std over seeds)

In [ ]:
RESULTS = f'{DRIVE}/results/s2'
os.makedirs(RESULTS, exist_ok=True)
table = summarize(runs_df)
if table.empty:
    print('no finished runs yet')
else:
    for ds, t in table.groupby('dataset'):
        print(f'\n{ds} (metric: {", ".join(t.metric.dropna().unique())})')
        display(t[['arch', 'attn', 'alpha', 'n', 'test (mean ± std)', 'val_mean', 'hours']])
    runs_df.to_csv(f'{RESULTS}/s2_runs.csv', index=False)
    table.to_csv(f'{RESULTS}/s2_summary.csv', index=False)
    print('saved', f'{RESULTS}/s2_runs.csv', 'and', f'{RESULTS}/s2_summary.csv')

## 10. Plot: test metric vs α per dataset, with mlp and gcn baselines

In [ ]:
import matplotlib.pyplot as plt

if not table.empty:
    datasets = sorted(table['dataset'].dropna().unique())
    fig, axes = plt.subplots(1, len(datasets), figsize=(4.2 * len(datasets), 3.6), squeeze=False)
    for ax, ds in zip(axes[0], datasets):
        t = table[table['dataset'] == ds]
        curve = t[~t['arch'].isin(BASELINES)].sort_values('alpha')
        for (arch, attn), c in curve.groupby(['arch', 'attn']):
            ax.errorbar(c['alpha'], c['mean'], yerr=c['std'], marker='o', capsize=3, label=f'{arch} ({attn})')
        for base, color in zip(BASELINES, ('#888780', '#D85A30')):
            b = t[t['arch'] == base]
            if not b.empty:
                m, s = b['mean'].iloc[0], b['std'].iloc[0]
                ax.axhline(m, color=color, ls='--', label=f'{base} (n={int(b["n"].iloc[0])})')
                ax.axhspan(m - s, m + s, color=color, alpha=0.12, lw=0)
        ax.set(title=ds, xlabel='α (0 = local, 1 = global)', ylabel=f'test {t["metric"].iloc[0]}')
        ax.set_xticks([0, 0.25, 0.5, 0.75, 1])
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(f'{RESULTS}/s2_alpha_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('saved', f'{RESULTS}/s2_alpha_curves.png')